In [1]:
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud
from nuscenes.utils.geometry_utils import transform_matrix , Quaternion 
import numpy as np
import open3d as o3d


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [44]:
# Load dataset
nusc = NuScenes(version="v1.0-mini", dataroot="v1.0-mini", verbose=True)
#Choose a scene
scene = nusc.scene[7]  # Select a specific scene
first_sample = nusc.get("sample", scene["first_sample_token"])

def compute_difference(pc1, pc2, voxel_size):
    # Voxelize both point clouds
    # Extract point arrays
    points1 = np.asarray(pc1.points)
    points2 = np.asarray(pc2.points)
    # Compute the difference by finding points in pc1 not in pc2 and vice versa
    # Find points in pc1 that are not in pc2 and vice versa
    dynamic_points = []
    # Check points in pc1 that are not in pc2
    for point in points1:
       
        if not any(np.allclose(point, other_point  ) for other_point in points2):
            dynamic_points.append(point)
    
    # Check points in pc2 that are not in pc1
    for point in points2:
        if not any(np.allclose(point, other_point) for other_point in points1):
            dynamic_points.append(point)
    
    return np.array(dynamic_points)



Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.431 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


In [45]:
# # Initialize aggregated point cloud
aggregated_pcd = o3d.geometry.PointCloud()
current_sample = first_sample
lidar_token = current_sample["data"]["LIDAR_TOP"]
lidar_data  = nusc.get("sample_data", lidar_token)
ego_pose    = nusc.get("ego_pose", lidar_data["ego_pose_token"])
calibrated_sensor = nusc.get(
    "calibrated_sensor", lidar_data["calibrated_sensor_token"]
)
# # Load lidar data
lidar_filepath = nusc.get_sample_data_path(lidar_token)
pointcloud = LidarPointCloud.from_file(lidar_filepath)

# voxelize the point cloud
voxel_size = 1
points = pointcloud.points[:3, :].T  # Take only the x, y, z coordinates and transposew
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd = pcd.voxel_down_sample(voxel_size=voxel_size)
# visualize the voxlized and orginal poit cloud 
# o3d.visualization.draw_geometries([pcd])

next_sample_token = current_sample["next"]


In [46]:
# Iterate over samples in the scene
current_sample = first_sample
print(len(current_sample))
index =0
voxel_size = 1000
dynamic_object = o3d.geometry.PointCloud()
while current_sample:
    print("current_sample------------------------" )
    lidar_token = current_sample["data"]["LIDAR_TOP"]
    lidar_data  = nusc.get("sample_data", lidar_token)
    ego_pose    = nusc.get("ego_pose", lidar_data["ego_pose_token"])
    calibrated_sensor = nusc.get(
        "calibrated_sensor", lidar_data["calibrated_sensor_token"]
    )
    #Load lidar data
    lidar_filepath = nusc.get_sample_data_path(lidar_token)
    pointcloud = LidarPointCloud.from_file(lidar_filepath)
    q = Quaternion(calibrated_sensor["rotation"])
    lidar_to_base = np.eye(4)
    lidar_to_base[:3, :3] = q.rotation_matrix
    lidar_to_base[:3, 3] = calibrated_sensor["translation"]
    pointcloud.transform(lidar_to_base)
    #Transform to global frame
    base_to_global = np.eye(4)
    q = Quaternion(ego_pose["rotation"])
    base_to_global[:3, :3] = q.rotation_matrix
    base_to_global[:3, 3] = ego_pose["translation"] 
    pointcloud.transform(base_to_global)
    #Add points to aggregated point cloud
    pcd = o3d.geometry.PointCloud()
    points = pointcloud.points[:3, :].T  # Take only the x, y, z coordinates and transpose
    pcd.points = o3d.utility.Vector3dVector(points)
    # pcd = pcd.voxel_down_sample(voxel_size=voxel_size)
    if index == 0 :
        prev_pcd = pcd
    else:
        # Compute the difference between the current and previous point cloud
        pcd_diff = o3d.geometry.PointCloud()
        dynamic_points = compute_difference(prev_pcd, pcd, voxel_size)
        pcd_diff.points = o3d.utility.Vector3dVector(dynamic_points)
        dynamic_object += pcd_diff
        
        
        # pcd_diff.points = o3d.utility.Vector3dVector(dynamic_points)
        # dynamic_object += pcd_diff
        prev_pcd = pcd
        # break
    
    aggregated_pcd += pcd
    # o3d.visualization.draw_geometries([pcd])
    # Move to next sample
    current_sample = (
        nusc.get("sample", current_sample["next"]) if current_sample["next"] else None
    )

    index += 1
    # if index == 2:
    #     break

# # Save aggregated point cloud 

# visualize aggregated point cloud
o3d.visualization.draw_geometries([aggregated_pcd])
# o3d.visualization.draw_geometries([dynamic_object])


7
current_sample------------------------
current_sample------------------------


KeyboardInterrupt: 

In [50]:
sample_cloud1 = o3d.geometry.PointCloud()
sample_cloud2 = o3d.geometry.PointCloud()

dynamic_points = np.array([[0, 0, 0] , [0.5 , 0.5 , 0 ], [1, 1, 0], 
                           [1.5,1.5,0] , [2, 2, 0] , [2.5,2.5,0] ,
                           [3, 3, 0],  [3.5,3.5,0] , [4.5, 4.5, 0] ,
                           [5.5, 5.5, 0],[6.6, 6.6, 0] , [7, 7, 0]
                           ,[8.5,8.5,0] , [4, 4, 0] , [5, 5, 0], 
                            [6, 6, 0] , [7, 7, 0] , [8, 8, 0] 
                           , [9, 9, 0], [10, 10, 00] , [11, 11, 0] ])
dynamic_points2 = np.array([[20, 20, 0] , [21 , 21 , 0 ], [21.5, 21.5, 0], 
                           [21.5,1.5,0] , [22.5,2.5,0] ,
                           [23, 32, 0],  [23.5,3.5,0] , [4.5, 4.5, 0] ,
                           [52.5, 25.5, 0],[6.6, 6.6, 0] , [27, 27, 0]
                           ,[28.25,28.5,0] , [24, 24, 0] , [25, 25, 0], 
                            [26, 26, 0] , [27, 27, 0] , [28, 28, 0] 
                           , [29,29, 0], [10, 10, 00] , [12, 12, 0] ])
sample_cloud1.points = o3d.utility.Vector3dVector(dynamic_points)
sample_cloud2.points = o3d.utility.Vector3dVector(dynamic_points2)

voxel_size = 1
pcd = sample_cloud1
# centroid voxel grid 

sample_cloud1 = pcd.voxel_down_sample(voxel_size=voxel_size)
voxel_points1 = np.asarray(sample_cloud1.points)
sample_cloud2 = sample_cloud2.voxel_down_sample(voxel_size=voxel_size)
voxel_points2 = np.asarray(sample_cloud2.points)
voxel_points1 = np.asarray(pcd.points)

p1_voxels = set(
                    map(tuple, np.floor(voxel_points1 / voxel_size).astype(int))
                )
p2_voxels = set(
                    map(tuple, np.floor(voxel_points2 / voxel_size).astype(int))
                )
print("p1_voxels" ,sorted(p1_voxels))
print("p2_voxels" , sorted((p2_voxels)))
# compute the difference between the two point clouds
dynamic_points = []
# Check points in pc1 that are not in pc2
for point in voxel_points1:
    if tuple(np.floor(point / voxel_size).astype(int)) not in p2_voxels:
        dynamic_points.append(point)
# Check points in pc2 that are not in pc1
for point in voxel_points2:
    if tuple(np.floor(point / voxel_size).astype(int)) not in p1_voxels:
        dynamic_points.append(point)

dynamic_points = np.array(dynamic_points)
print(dynamic_points)
dynamic_voxels = p1_voxels and p2_voxels
print(dynamic_voxels)


p1_voxels [(0, 0, 0), (1, 1, 0), (2, 2, 0), (3, 3, 0), (4, 4, 0), (5, 5, 0), (6, 6, 0), (7, 7, 0), (8, 8, 0), (9, 9, 0), (10, 10, 0), (11, 11, 0)]
p2_voxels [(4, 4, 0), (6, 6, 0), (10, 10, 0), (12, 12, 0), (20, 20, 0), (21, 1, 0), (21, 21, 0), (22, 2, 0), (23, 3, 0), (23, 32, 0), (24, 24, 0), (25, 25, 0), (26, 26, 0), (27, 27, 0), (28, 28, 0), (29, 29, 0), (52, 25, 0)]
[[ 0.     0.     0.   ]
 [ 0.5    0.5    0.   ]
 [ 1.     1.     0.   ]
 [ 1.5    1.5    0.   ]
 [ 2.     2.     0.   ]
 [ 2.5    2.5    0.   ]
 [ 3.     3.     0.   ]
 [ 3.5    3.5    0.   ]
 [ 5.5    5.5    0.   ]
 [ 7.     7.     0.   ]
 [ 8.5    8.5    0.   ]
 [ 5.     5.     0.   ]
 [ 7.     7.     0.   ]
 [ 8.     8.     0.   ]
 [ 9.     9.     0.   ]
 [11.    11.     0.   ]
 [22.5    2.5    0.   ]
 [21.5    1.5    0.   ]
 [23.    32.     0.   ]
 [23.5    3.5    0.   ]
 [52.5   25.5    0.   ]
 [29.    29.     0.   ]
 [20.    20.     0.   ]
 [27.    27.     0.   ]
 [12.    12.     0.   ]
 [21.25  21.25   0.   ]
 [28